# Price & Product Intelligence Analysis

This notebook performs the data preparation and SQL analysis stage of the **Price & Product Intelligence Tracker** project.

### Objectives
- Clean and standardise the scraped book data.
- Store the data in a SQLite database.
- Use SQL to generate product and pricing KPIs.
- Investigate price changes across the tracking period.
- Perform data-quality checks before dashboard reporting.

The SQL outputs generated here support the Power BI dashboard.

In [1]:
# Load the scraped book dataset for analysis.
import sqlite3
import pandas as pd
df = pd.read_csv("all_books.csv")

In [2]:
# Remove duplicate observations and load the cleaned dataset into the SQLite price-history table.
df = pd.read_csv("data/all_books.csv")
df = df.drop_duplicates(subset=["title", "price", "scrape_date"], keep="first")

conn = sqlite3.connect("data/books.db")
df.to_sql("price_history", conn, if_exists="replace", index=False)
conn.close()

print(df.shape)

(5000, 5)


In [3]:
# Convert the price from text to number and remove '£' symbol
df['price'] = df['price'].str.replace('£', '', regex=False).astype(float)

In [4]:
# Convert the rating from word to number
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df['rating'] = df['rating'].map(rating_map)

In [5]:
# Confirming the cleaned work
print(df[['price', 'rating']].dtypes)
print(df[['price', 'rating']].head())

price     float64
rating      int64
dtype: object
   price  rating
0  51.77       3
1  53.74       1
2  50.10       1
3  47.82       4
4  54.23       5


In [6]:
df.head(5)

,title,price,availability,rating,scrape_date
0,A Light in the Attic,51.77,In stock,3,2026-08-07
1,Tipping the Velvet,53.74,In stock,1,2026-08-07
2,Soumission,50.10,In stock,1,2026-08-07
3,Sharp Objects,47.82,In stock,4,2026-08-07
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,2026-08-07


**SQL Queries**

In [7]:
# Load the cleaned dataset into the SQLite database for further SQL analysis
conn = sqlite3.connect("data/books.db")
df.to_sql("price_history", conn, if_exists="replace", index=False)
conn.close()

In [8]:
# Run a sample query to confirm the price-history table can be queried
conn = sqlite3.connect("data/books.db")

query = "SELECT * FROM price_history LIMIT 5"
result = pd.read_sql(query, conn)
print(result)

conn.close()

                                   title  price availability  rating  \
0                   A Light in the Attic  51.77     In stock       3   
1                     Tipping the Velvet  53.74     In stock       1   
2                             Soumission  50.10     In stock       1   
3                          Sharp Objects  47.82     In stock       4   
4  Sapiens: A Brief History of Humankind  54.23     In stock       5   

  scrape_date  
0  2026-08-07  
1  2026-08-07  
2  2026-08-07  
3  2026-08-07  
4  2026-08-07  


### Total Number of books tracked

In [9]:
# Count the distinct book titles tracked
conn = sqlite3.connect("data/books.db")

query = """
SELECT COUNT(DISTINCT title) AS total_books
FROM price_history
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q1_total_books.csv", index=False)

conn.close()

   total_books
0          999


In [10]:
# Identify titles with more than one recorded price
conn = sqlite3.connect("data/books.db")

query = """
SELECT title, COUNT(DISTINCT price) AS distinct_prices, COUNT(*) AS total_rows
FROM price_history
GROUP BY title
HAVING COUNT(DISTINCT price) > 1
"""
result = pd.read_sql(query, conn)
print(result)

conn.close()

                    title  distinct_prices  total_rows
0  The Star-Touched Queen                2          10


### First and Last Scrape date

In [11]:
# Calculate the number of scrape days and identify the earliest and latest dates
conn = sqlite3.connect("data/books.db")

query = """
SELECT COUNT(DISTINCT scrape_date) AS total_scrape_days,
       MIN(scrape_date) AS earliest_date,
       MAX(scrape_date) AS latest_date
FROM price_history
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q2_scrape_date_range.csv", index=False)

conn.close()

   total_scrape_days earliest_date latest_date
0                  5    2026-08-07  2026-08-17


### No. of books currently out of stock (latest date)

In [12]:
# Count books that were out of stock on the latest scrape date
conn = sqlite3.connect("data/books.db")

query = """
SELECT COUNT(*) AS out_of_stock
FROM price_history
WHERE availability != 'In stock'
AND scrape_date = (SELECT MAX(scrape_date) FROM price_history)
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q3_out_of_stock_count.csv", index=False)

conn.close()

   out_of_stock
0             0


### Average book prices across the catalog (latest date)

In [13]:
# Calculate the latest average book price
conn = sqlite3.connect("data/books.db")

query = """
SELECT ROUND(AVG(price), 2) AS avg_price
FROM price_history
WHERE scrape_date = (SELECT MAX(scrape_date) FROM price_history)
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q4_avg_price.csv", index=False)

conn.close()

   avg_price
0      35.07


### Top 5 most expensive books in the catalog

In [14]:
# Identify the five most expensive books using the latest prices
conn = sqlite3.connect("data/books.db")

query = """
SELECT title, price, rating
FROM price_history
WHERE scrape_date = (SELECT MAX(scrape_date) FROM price_history)
ORDER BY price DESC
LIMIT 5
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q5_top5_expensive.csv", index=False)

conn.close()

                                title  price  rating
0  The Perfect Play (Play by Play #1)  59.99       3
1   Last One Home (New Beginnings #1)  59.98       3
2    Civilization and Its Discontents  59.95       2
3      The Barefoot Contessa Cookbook  59.92       5
4           The Diary of a Young Girl  59.90       3


### Top 5 least expensive books in the catalog

In [15]:
# Identify the five least expensive books using the latest prices
conn = sqlite3.connect("data/books.db")

query = """
SELECT title, price, rating
FROM price_history
WHERE scrape_date = (SELECT MAX(scrape_date) FROM price_history)
ORDER BY price ASC
LIMIT 5
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q5_top5_cheapest.csv", index=False)

conn.close()

                                               title  price  rating
0                         An Abundance of Katherines  10.00       5
1                              The Origin of Species  10.01       4
2  The Tipping Point: How Little Things Can Make ...  10.02       2
3                                           Patience  10.16       3
4                               Greek Mythic History  10.23       5


### Average price of books by star rating

In [16]:
# Compare average prices across star-rating groups
conn = sqlite3.connect("data/books.db")

query = """
SELECT rating, ROUND(AVG(price), 2) AS avg_price, COUNT(*) AS num_of_books
FROM price_history
WHERE scrape_date = (SELECT MAX(scrape_date) FROM price_history)
GROUP BY rating
ORDER BY rating
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q6_price_by_rating.csv", index=False)

conn.close()

   rating  avg_price  num_of_books
0       1      34.56           226
1       2      34.81           196
2       3      34.69           203
3       4      36.09           179
4       5      35.37           196


### Which specific books are currently out of stock?

In [17]:
# Return the individual books currently marked as out of stock
conn = sqlite3.connect("data/books.db")

query = """
SELECT title, price
FROM price_history
WHERE availability != 'In stock'
AND scrape_date = (SELECT MAX(scrape_date) FROM price_history)
"""
result = pd.read_sql(query, conn)
print(result)
print(f"Number of books which are currently out of stock: {len(result)}")
result.to_csv("data/q7_out_of_stock.csv", index=False)

conn.close()

Empty DataFrame
Columns: [title, price]
Index: []
Number of books which are currently out of stock: 0


### Which books have changed price since tracking began, and by how much?

In [18]:
# Compare earliest and latest prices to identify books with price changes
conn = sqlite3.connect("data/books.db")

query = """
SELECT first.title,
        first.price AS earliest_price,
        latest.price AS latest_price,
        ROUND((latest.price - first.price), 2) AS price_change,
        ROUND(((latest.price - first.price) / first.price * 100), 2) AS percent_change
FROM price_history first
JOIN price_history latest
    ON first.title = latest.title
WHERE first.scrape_date = (SELECT MIN(scrape_date) FROM price_history)
AND latest.scrape_date = (SELECT MAX(scrape_date) FROM price_history)
AND first.price != latest.price
AND first.title != 'The Star-Touched Queen'
ORDER BY ABS(price_change) DESC
"""
result = pd.read_sql(query, conn)
print(result)
result.to_csv("data/q8_price_changes.csv", index=False)

conn.close()

Empty DataFrame
Columns: [title, earliest_price, latest_price, price_change, percent_change]
Index: []


### Data Quality Check: Duplicate Title Investigation
Found a discrepancy in unique book counts. Investigating below.

In [19]:
# Investigate the duplicate-title data-quality issue
conn = sqlite3.connect("data/books.db")

query = """
SELECT title, price, scrape_date
FROM price_history
WHERE title = 'The Star-Touched Queen'
ORDER BY scrape_date, price
"""
result = pd.read_sql(query, conn)
print(result)

conn.close()

                    title  price scrape_date
0  The Star-Touched Queen  32.30  2026-08-07
1  The Star-Touched Queen  46.02  2026-08-07
2  The Star-Touched Queen  32.30  2026-08-12
3  The Star-Touched Queen  46.02  2026-08-12
4  The Star-Touched Queen  32.30  2026-08-14
5  The Star-Touched Queen  46.02  2026-08-14
6  The Star-Touched Queen  32.30  2026-08-15
7  The Star-Touched Queen  46.02  2026-08-15
8  The Star-Touched Queen  32.30  2026-08-17
9  The Star-Touched Queen  46.02  2026-08-17


In [20]:
# Data quality check: confirm row counts and unique titles are consistent across scrape dates
# (validates that drop_duplicates() in the loading step (Cell 2) worked correctly, and that
# each scrape day has the expected ~1000 rows with no lingering duplication)

conn = sqlite3.connect("data/books.db")

query = """
SELECT scrape_date, COUNT(*) AS total_rows, COUNT(DISTINCT title) AS unique_titles
FROM price_history
GROUP BY scrape_date
"""
result = pd.read_sql(query, conn)
print(result)

conn.close()

  scrape_date  total_rows  unique_titles
0  2026-08-07        1000            999
1  2026-08-12        1000            999
2  2026-08-14        1000            999
3  2026-08-15        1000            999
4  2026-08-17        1000            999


### Which books show the largest price increase and largest price decrease?

In [21]:
# Q9: Top 5 price increases and top 5 price decreases between earliest and latest scrape date.
# Note: "The Star-Touched Queen" is explicitly excluded — it corresponds to two distinct
# products that share an identical title (confirmed during data quality check above).
# Without a unique product ID, the self-join below cannot tell these two books apart,
# causing false "price changes" when one book's price is cross-matched against the other's.
conn = sqlite3.connect("data/books.db")

query_q9_increases = """
SELECT 
    first.title,
    first.price AS earliest_price,
    latest.price AS latest_price,
    ROUND(latest.price - first.price, 2) AS price_change
FROM price_history first
JOIN price_history latest
    ON first.title = latest.title
WHERE first.scrape_date = (SELECT MIN(scrape_date) FROM price_history)
AND latest.scrape_date = (SELECT MAX(scrape_date) FROM price_history)
AND first.price != latest.price
AND first.title != 'The Star-Touched Queen'
ORDER BY price_change DESC
LIMIT 5
"""
increases = pd.read_sql(query_q9_increases, conn)
increases.to_csv("data/q9_top_increases.csv", index=False)

query_q9_decreases = query_q9_increases.replace("ORDER BY price_change DESC", "ORDER BY price_change ASC")
decreases = pd.read_sql(query_q9_decreases, conn)
decreases.to_csv("data/q9_top_decreases.csv", index=False)

print("Top 5 price increases:")
print(increases)
print("\nTop 5 price decreases:")
print(decreases)

conn.close()

Top 5 price increases:
Empty DataFrame
Columns: [title, earliest_price, latest_price, price_change]
Index: []

Top 5 price decreases:
Empty DataFrame
Columns: [title, earliest_price, latest_price, price_change]
Index: []
